# ERCOT Texas Synthetic Grid — Substation Planning Notebook

---

## Pipeline Overview

| Step | Cell | Description |
|------|------|-------------|
| 1 | Imports | Load Python packages |
| 2 | Init SGB | Create `SyntheticGridBuilder` from config |
| 3 | Paths | Set Input/Output directory paths |
| 4 | Read Data | Census (Gazetteer + DP1), EIA 860, ERCOT GIS Report |
| 5 | Shapefiles | Weather zones, counties, coastline |
| 6 | Helpers | Spatial assignment functions + zone-region mapping |
| 7 | Reference Case | Extract kV levels, load shares, and load voltage distribution from PowerWorld |
| 8 | Load Fragments | Census tracts to load fragments (~6,200) |
| 9a | Gen Fragments (EIA) | EIA 860 operable plants to gen fragments |
| 9b | Gen Fragments (GIS) | ERCOT GIS planned generators (IA Signed) + combined fuel type table |
| 10 | Large Loads | Synthetic large load allocation (>= 75 MW) |
| 11 | Validation Plot | Ripley's L, NN CDF, spatial map |
| 12 | Pre-processing | Reactive power, generator parameters, costs |
| 13 | Areas & Zones | Create Area (weather zone) and Zone (county) objects |
| 14 | Clustering | Merge load fragments into substations |
| 15 | EHV + kV | KMeans EHV selection + voltage level + Delaunay step-down + 765 kV corridors + parallel buses |
| 15b | Visualization | 765 kV corridor and weather zone maps |
| 16 | Bus Creation & Export | Buses, load voltage allocation, large-load subs, DataCenter labeling, census scaling, AUX export |

## Key Tunable Parameters

| Parameter | Cell | Default | Effect |
|-----------|------|---------|--------|
| `cluster_load_frags(N)` | 14 | 5,000 | Number of core substations. Increase for more subs/buses. |
| `FINAL_BUS_TARGET` | 15 | 10,000 | Approximate total bus count. Parallel buses fill half the gap. |
| `ehv_target_ratio` | 15 | ~0.154 | Fraction of buses at 345 kV (from reference case). |
| `TARGET_69_RATIO` | 15 | 0.15 | Fraction of buses at 69 kV. |
| Delaunay probability | 15 | 30% / 40% | Chance of adding step-down bus from neighbor (non-EHV / EHV). |
| `N_LARGE_LOADS` | 10 | 250 | Number of synthetic large loads (>= 75 MW). |
| `MIN_345_LOAD_MW` | 16 | 50 | Minimum MW for a load on a 345 kV bus. Below this → 138 kV. |
| `SPLIT_FRACTION` | 16 | 138: 1.2%, 345: 16.7% | Fraction of highest-MW subs with split loads. |
| `DC_PROXIMITY_KM` | 16 | 50 km | Max distance to label a large-load sub as DataCenter. |
| `TARGET_SYSTEM_LOAD_MW` | 16 | 145,000 | Total system load target (2031 ERCOT CDR). |
| `REFERENCE_CASE` | 7 | (path to .pwb) | PowerWorld reference case for kV ratios and load shares. |


# Cell 1: Imports

Loads all required Python packages: `sklearn` (KMeans), `geopandas`/`shapely` (spatial ops), `esa.SAW` (PowerWorld), `sgb_suite` (Synthetic Grid Builder), `pandas`/`numpy`, and `large_load_allocation`.


In [ ]:
import os
from time import time
from numpy.random import random

from sklearn.cluster import KMeans
import geopandas as gpd
from shapely.geometry import Point

from gridworkbench.containers import Bus, Node
from gridworkbench.devices import Load
from gridworkbench.utils import geograph

from sgb_suite.syntheticgridbuilder import SyntheticGridBuilder
from sgb_suite.sub_planning import LoadFragment
from sgb_suite.sub_planning import GenFragment, GenUnit
import pandas as pd
import numpy as np
from esa import SAW

from large_load_allocation import run_large_load_allocation, LARGE_LOAD_MW

tlap = time()
def lap(txt): global tlap; print(txt + f" | {time()-tlap} secs"); tlap = time()

# Cell 2: Initialize Synthetic Grid Builder

Creates a `SyntheticGridBuilder` instance from `texas_config.txt`. Output folder is overridden in the next cell.


In [ ]:
sgb = SyntheticGridBuilder("texas_config.txt")

# Configuration — Read Before Running

This notebook uses **relative paths** from its own location. The only hardcoded path is `REFERENCE_CASE` in Cell 7.

```
Texas_Synthetic_Grid/
├── Creation/          ← this notebook
├── Inputs/            ← all input files
└── Output/            ← ERCOT_Subs.aux saved here
```

| Item | Where | Notes |
|------|-------|-------|
| Census Gazetteer + DP1 | `Inputs/` | From Census Bureau |
| EIA 860 Plant/Gen | `Inputs/` | From eia.gov |
| ERCOT GIS Report | `Inputs/` | Latest ERCOT GIS queue report |
| PowerWorld reference | Cell 7 | Update `REFERENCE_CASE` path |


# Data Sources — Download Links

| Data | URL |
|------|-----|
| Census Gazetteer | https://www.census.gov/geographies/reference-files/time-series/geo/gazetteer-files.html |
| Census DP1 | https://data.census.gov (search DP1, select Texas Census Tracts) |
| EIA 860 | https://www.eia.gov/electricity/data/eia860/ |


In [ ]:
PROJECT_DIR = os.getcwd()
FOLDER_DIR = os.path.dirname(PROJECT_DIR)
INPUT_DIR = os.path.join(FOLDER_DIR, "Inputs")
OUTPUT_DIR = os.path.join(FOLDER_DIR, "Output")

sgb.input_folder = INPUT_DIR
sgb.output_folder = OUTPUT_DIR
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Input  folder: {INPUT_DIR}")
print(f"Output folder: {OUTPUT_DIR}")

# Cell 3: Set Input/Output Paths

Derives all paths relative to the notebook location. Overrides `sgb.input_folder` and `sgb.output_folder`.


In [ ]:
# ── Read Input Files with Pandas ─────────────────────────────────────────────

# Gazetteer: national census tracts — filter to TX, match by GEOID
census2020_geo_df = pd.read_csv(
    os.path.join(sgb.input_folder, "2025_Gaz_tracts_national.csv"),
    dtype={'GEOID': str, 'USPS': str}
)
census2020_geo_df = census2020_geo_df[census2020_geo_df['USPS'] == 'TX'].copy()
print(f"Gazetteer: {len(census2020_geo_df)} Texas census tracts")

# DP1 Population: Texas census tracts only (2nd row is human-readable labels — skip it)
census2020_pop_df = pd.read_csv(
    os.path.join(sgb.input_folder, "DECENNIALDP2020.DP1-Data.csv"),
    skiprows=[1], dtype={'GEO_ID': str}
)
# GEO_ID format: "1400000US48XXX..." → extract 11-digit GEOID
census2020_pop_df['GEOID'] = census2020_pop_df['GEO_ID'].str.replace('1400000US', '', regex=False)
census2020_pop_df = census2020_pop_df[census2020_pop_df['GEOID'].str.len() == 11].copy()
print(f"DP1: {len(census2020_pop_df)} Texas census tracts")

# Merge geo + pop on GEOID — single iterable with all needed columns
census_df = census2020_geo_df.merge(census2020_pop_df, on='GEOID', how='inner')
census_records = census_df.to_dict('records')
print(f"Merged: {len(census_records)} matched census tracts")

# EIA 860 Plant data — row 0 is title, row 1 is actual headers
eia860_2022_plants = pd.read_excel(
    os.path.join(sgb.input_folder, "2___Plant_Y2024.xlsx"),
    sheet_name='Plant', header=1
).to_dict('records')
print(f"EIA Plants: {len(eia860_2022_plants)} | columns sample: {list(pd.read_excel(os.path.join(sgb.input_folder, '2___Plant_Y2024.xlsx'), sheet_name='Plant', header=1, nrows=0).columns[:5])}")

# EIA 860 Generator data — row 0 is title, row 1 is actual headers
eia860_2022_gens_operable = pd.read_excel(
    os.path.join(sgb.input_folder, "3_1_Generator_Y2024.xlsx"),
    sheet_name='Operable', header=1
).to_dict('records')
print(f"EIA Operable Generators: {len(eia860_2022_gens_operable)} | columns sample: {list(pd.read_excel(os.path.join(sgb.input_folder, '3_1_Generator_Y2024.xlsx'), sheet_name='Operable', header=1, nrows=0).columns[:5])}")

# ERCOT GIS Report — Large Gen + Small Gen sheets (planned/queued generation)
_gis_file = os.path.join(sgb.input_folder, "RPT.00015933.0000000000000000.20260202.150301327.GIS_Report_January2026.xlsx")
_large = pd.read_excel(_gis_file, sheet_name='Project Details - Large Gen', header=0)
_small = pd.read_excel(_gis_file, sheet_name='Project Details - Small Gen', header=0)
eia860_2022_gens_planned = pd.concat([_large, _small], ignore_index=True)
print(f"ERCOT GIS planned generators: {len(eia860_2022_gens_planned)} ({len(_large)} large + {len(_small)} small)")

# Cell 4: Read Input Files

Loads Census Gazetteer + DP1 (merged on GEOID), EIA 860 Plant/Generator data, and ERCOT GIS Report using pandas.


# Data Sources — Shapefiles

| Shapefile | Source |
|-----------|--------|
| Coastline (10m) | https://www.naturalearthdata.com/downloads/10m-physical-vectors/10m-coastline/ |
| USGS NHD Lakes (optional) | https://www.usgs.gov/national-hydrography/access-national-hydrography-products |


In [ ]:
# ── Load Shapefiles ──────────────────────────────────────────────────────────

# Weather zones (Area level) — 8 ERCOT weather zones
wz_gdf = gpd.read_file(os.path.join(sgb.input_folder, "ERCOT_WEATHER_ZONE_modified.shp"))
wz_gdf = wz_gdf.to_crs('epsg:4326')
ercot_zones_gdf = wz_gdf[wz_gdf['ERCOT_Clim'].notna()]  # exclude non-ERCOT areas
ercot_boundary = ercot_zones_gdf.union_all()  # merged ERCOT boundary polygon

# Counties (Zone level) — 254 Texas counties, each assigned to a weather zone
county_gdf = gpd.read_file(os.path.join(sgb.input_folder, "Texas_Counties.shp"))
county_gdf = county_gdf.to_crs('epsg:4326')

# Coastline
db_coasts = gpd.read_file(os.path.join(sgb.input_folder, "ne_10m_coastline", "ne_10m_coastline.shp"))
coasts = [db_coasts.iloc[i]["geometry"] for i in range(db_coasts.shape[0])]

# Lakes (empty for now — add NHD Texas data if available)
lake_bounds = []

print(f"Loaded {len(ercot_zones_gdf)} weather zones, {len(county_gdf)} counties")
print(f"Weather zones: {sorted(ercot_zones_gdf['ERCOT_Clim'].tolist())}")

# Cell 5: Load Shapefiles

Loads weather zone (`ERCOT_Clim`), county (`US_Count_1`), and coastline shapefiles. Builds the merged `ercot_boundary` polygon for filtering points to ERCOT footprint.


In [ ]:
# ── Helper Functions: Assign Area (Weather Zone) and Zone (County) ────────────

def assign_weather_zone(lat, lon):
    """Assign ERCOT weather zone (Area) using shapefile geometry."""
    point = Point(lon, lat)
    for _, row in ercot_zones_gdf.iterrows():
        if row['geometry'].contains(point):
            return row['ERCOT_Clim']
    return None  # outside ERCOT

def assign_county(lat, lon):
    """Assign Texas county (Zone) using shapefile geometry."""
    point = Point(lon, lat)
    for _, row in county_gdf.iterrows():
        if row['geometry'].contains(point):
            return row['US_Count_1'].replace(' County', '').strip()
    return None

def is_inside_ercot(lat, lon):
    """Check if a point is inside the ERCOT boundary."""
    return ercot_boundary.contains(Point(lon, lat))

# ── Build zone_regions (county → weather zone) from shapefiles ────────────────
# Uses county centroids to determine which weather zone each county belongs to
zone_regions = {}
for _, row in county_gdf.iterrows():
    county_name = row['US_Count_1'].replace(' County', '').strip()
    centroid = row['geometry'].centroid
    wz = assign_weather_zone(centroid.y, centroid.x)
    if wz:
        zone_regions[county_name] = wz

# area_regions: weather_zone → itself
area_regions = {wz: wz for wz in sorted(ercot_zones_gdf['ERCOT_Clim'].tolist())}

print(f"Built zone_regions: {len(zone_regions)} counties mapped to weather zones")
print(f"Weather zone distribution:")
from collections import Counter
wz_counts = Counter(zone_regions.values())
for wz in sorted(wz_counts):
    print(f"  {wz:20s}: {wz_counts[wz]} counties")

# Cell 6: Helper Functions and Zone-Region Mapping

Defines `assign_weather_zone()`, `assign_county()`, `is_inside_ercot()` via point-in-polygon, and builds `zone_regions` (county → weather zone mapping).


In [ ]:
# ── Auto-Extract kV + Load Distribution from PowerWorld Reference Case ────────

# REFERENCE_CASE = r"<REDACTED_PATH>\Cases Files\Synthetic_grid\Texas2k_series24_cases_with_dynamics\Texas2k_series25_case1_summerpeak\Texas2k_series25_case1_summerpeak_weather_county.pwb"
REFERENCE_CASE = r"<REDACTED_PATH>\Cases Files\ERCOT\ver_30U1_2025-11-14\ERCOT-SS-25SSWG_2026_SUM1_U1_OnPeak_Pass_Final_11132025_Set3-PWver24.pwb"
saw = SAW(REFERENCE_CASE)

# ── 1. Substations: coordinates + kV levels ──────────────────────────────────
ref_subs = saw.GetParametersMultipleElement('Substation',
    ['SubNum', 'BGNominalkvRange:1', 'Latitude', 'Longitude'])
ref_subs['Latitude'] = pd.to_numeric(ref_subs['Latitude'], errors='coerce')
ref_subs['Longitude'] = pd.to_numeric(ref_subs['Longitude'], errors='coerce')
ref_subs = ref_subs.dropna(subset=['Latitude', 'Longitude']).copy()
print(f"Reference case: {len(ref_subs)} substations with coordinates")

# ── 2. Spatial join: assign weather zone + county from shapefiles ─────────────
sub_gdf = gpd.GeoDataFrame(
    ref_subs,
    geometry=gpd.points_from_xy(ref_subs['Longitude'], ref_subs['Latitude']),
    crs='epsg:4326'
)
# Weather zone (inner = ERCOT subs only)
sub_gdf = (gpd.sjoin(sub_gdf, ercot_zones_gdf[['ERCOT_Clim', 'geometry']],
                      how='inner', predicate='within')
           .drop(columns='index_right')
           .drop_duplicates('SubNum', keep='first')
           .rename(columns={'ERCOT_Clim': 'weather_zone'}))
# County
sub_gdf = (gpd.sjoin(sub_gdf, county_gdf[['US_Count_1', 'geometry']],
                      how='left', predicate='within')
           .drop(columns='index_right', errors='ignore')
           .drop_duplicates('SubNum', keep='first'))
sub_gdf['county'] = sub_gdf['US_Count_1'].str.replace(' County', '').str.strip()

ref_subs = pd.DataFrame(sub_gdf.drop(columns=['geometry', 'US_Count_1'], errors='ignore'))
print(f"  Inside ERCOT: {len(ref_subs)} subs, {ref_subs['county'].notna().sum()} with county")

# ── 3. Bus kV counts from reference case ─────────────────────────────────────
ref_buses_all = saw.GetParametersMultipleElement('Bus', ['BusNum', 'BusNomVolt', 'SubNum'])
ref_buses_all['BusNomVolt'] = pd.to_numeric(ref_buses_all['BusNomVolt'], errors='coerce')
ref_kv_counts_all = (ref_buses_all[ref_buses_all['BusNomVolt'] >= 69]['BusNomVolt']
                     .astype(int).value_counts().sort_index())

# Only keep kV levels with >= 1000 buses
ref_kv_counts = ref_kv_counts_all[ref_kv_counts_all >= 1000]
ref_total_buses = ref_kv_counts.sum()
ALLOWED_KVS = set(ref_kv_counts.index)

ref_345_ratio = ref_kv_counts.get(345, 0) / ref_total_buses
ref_69_ratio = ref_kv_counts.get(69, 0) / ref_total_buses
ref_138_ratio = ref_kv_counts.get(138, 0) / ref_total_buses

print(f"\nReference bus kV distribution (>= 1000 buses):")
for kv, count in ref_kv_counts.items():
    print(f"  {kv:>6} kV: {count:>5} buses ({100*count/ref_total_buses:.1f}%)")
print(f"  Total: {ref_total_buses} buses")
print(f"  345 kV ratio (for EHV target): {ref_345_ratio:.3f}")
print(f"  69 kV ratio: {ref_69_ratio:.3f}")
print(f"  138 kV ratio: {ref_138_ratio:.3f}")

# ── 4. Build kv_zones / kv_areas (filtered to ALLOWED_KVS) ───────────────────
def parse_all_kvs(kv_str):
    try:
        return sorted(set(int(float(x)) for x in str(kv_str).split() if float(x) >= 69))
    except:
        return [138]

ref_subs['all_kvs'] = ref_subs['BGNominalkvRange:1'].apply(parse_all_kvs)

county_kvs, area_kvs = {}, {}
for _, row in ref_subs.iterrows():
    county_kvs.setdefault(row['county'], set()).update(row['all_kvs'])
    area_kvs.setdefault(row['weather_zone'], set()).update(row['all_kvs'])

kv_zones = {c: sorted(kv for kv in kvs if kv in ALLOWED_KVS) or [138]
            for c, kvs in county_kvs.items()}
kv_areas = {wz: sorted(kv for kv in kvs if kv in ALLOWED_KVS) or [138]
            for wz, kvs in area_kvs.items()}

# Update zone_regions with any counties not already mapped from shapefiles
for _, row in ref_subs.drop_duplicates('county').iterrows():
    if row['county'] not in zone_regions and row['weather_zone']:
        zone_regions[row['county']] = row['weather_zone']

# ── 5. Per-zone load shares ──────────────────────────────────────────────────
ref_loads_df = saw.GetParametersMultipleElement('Load', ['BusNum', 'LoadSMW'])
ref_loads_df = (ref_loads_df
    .merge(ref_buses_all[['BusNum', 'SubNum']], on='BusNum')
    .merge(ref_subs[['SubNum', 'weather_zone']].drop_duplicates('SubNum'), on='SubNum'))
ref_loads_df['LoadSMW'] = pd.to_numeric(ref_loads_df['LoadSMW'], errors='coerce').fillna(0)

zone_load_totals = ref_loads_df.groupby('weather_zone')['LoadSMW'].sum()
ref_zone_load_shares = (zone_load_totals / zone_load_totals.sum()).to_dict()

print(f"\nPer-zone load distribution:")
for wz in sorted(ref_zone_load_shares):
    print(f"  {wz:20s}: {ref_zone_load_shares[wz]:.3f} ({zone_load_totals[wz]:.0f} MW)")
print(f"  {'TOTAL':20s}: {zone_load_totals.sum():.0f} MW")


# ── 6. Load distribution by lowest substation voltage level ───────────────────
# For subs with load > 0, check the lowest kV level (BGNominalkvRange:2)
ref_subs_ext = saw.GetParametersMultipleElement('Substation',
    ['SubNum', 'BGNominalkvRange:1', 'BGNominalkvRange:2'])

# Aggregate load per substation
sub_load_totals = (ref_loads_df.groupby('SubNum')['LoadSMW'].sum()
                   .reset_index()
                   .rename(columns={'LoadSMW': 'TotalLoadMW'}))

ref_subs_ext = ref_subs_ext.merge(sub_load_totals, on='SubNum', how='left')
ref_subs_ext['TotalLoadMW'] = ref_subs_ext['TotalLoadMW'].fillna(0)

# Parse lowest kV
def parse_lowest_kv(kv_str):
    try:
        kvs = [int(float(x)) for x in str(kv_str).split() if float(x) > 0]
        return min(kvs) if kvs else None
    except:
        return None

ref_subs_ext['LowestKV'] = ref_subs_ext['BGNominalkvRange:2'].apply(parse_lowest_kv)

# Filter to subs with load and lowest kV >= 69 (transmission only)
subs_with_load = ref_subs_ext[(ref_subs_ext['TotalLoadMW'] > 0) & (ref_subs_ext['LowestKV'] >= 69)].copy()

print(f"Substations with load > 0: {len(subs_with_load)} / {len(ref_subs_ext)}")
print(f"Load distribution by lowest substation kV level:")
kv_groups = subs_with_load.groupby('LowestKV').agg(
    Subs=('SubNum', 'count'),
    TotalMW=('TotalLoadMW', 'sum')
).sort_index()
for kv, row in kv_groups.iterrows():
    pct_subs = 100 * row['Subs'] / len(subs_with_load)
    pct_mw = 100 * row['TotalMW'] / subs_with_load['TotalLoadMW'].sum()
    level = "distribution" if kv < 69 else ("sub-transmission" if kv < 100 else "transmission")
    print(f"  {kv:>6} kV ({level:15s}): {int(row['Subs']):>5} subs ({pct_subs:5.1f}%), {row['TotalMW']:>10,.0f} MW ({pct_mw:5.1f}%)")
print(f"  {'TOTAL':>24s}: {len(subs_with_load):>5} subs,       {subs_with_load['TotalLoadMW'].sum():>10,.0f} MW")

# --- 7. Load voltage distribution by bus kV vs substation max kV
ref_buses = saw.GetParametersMultipleElement('Bus', ['BusNum', 'BusNomVolt', 'SubNum'])
ref_buses['BusNomVolt'] = pd.to_numeric(ref_buses['BusNomVolt'], errors='coerce')
ref_buses['SubNum'] = ref_buses['SubNum'].astype(str)

ref_loads_v = saw.GetParametersMultipleElement('Load', ['BusNum', 'LoadSMW'])
ref_loads_v['LoadSMW'] = pd.to_numeric(ref_loads_v['LoadSMW'], errors='coerce').fillna(0)
ref_loads_v['BusNum'] = ref_loads_v['BusNum'].astype(str)
ref_buses['BusNum'] = ref_buses['BusNum'].astype(str)

# Join Load -> Bus -> Substation
# Filter out buses below 69 kV
ref_buses = ref_buses[ref_buses['BusNomVolt'] >= 69]
ref_loads_v = ref_loads_v.merge(ref_buses[['BusNum', 'BusNomVolt', 'SubNum']], on='BusNum', how='left')

# Get max kV per substation
ref_subs_kv = ref_subs_ext.copy()
ref_subs_kv['SubNum'] = ref_subs_kv['SubNum'].astype(str)
def _parse_max_kv(kv_str):
    try:
        kvs = [float(x) for x in str(kv_str).split() if float(x) > 0]
        return max(kvs) if kvs else None
    except:
        return None
ref_subs_kv['MaxKV'] = ref_subs_kv['BGNominalkvRange:1'].apply(_parse_max_kv)

ref_loads_v = ref_loads_v.merge(ref_subs_kv[['SubNum', 'MaxKV']], on='SubNum', how='left')
ref_loads_v = ref_loads_v[(ref_loads_v['MaxKV'].notna()) & (ref_loads_v['BusNomVolt'] >= 69) & (ref_loads_v['MaxKV'] >= 69)]

# Bucket kV levels: map to standard levels (69, 138, 345)
def _bucket_kv(kv):
    if kv <= 69: return 69
    elif kv <= 200: return 138
    else: return 345

ref_loads_v['BusKVBucket'] = ref_loads_v['BusNomVolt'].apply(_bucket_kv)
ref_loads_v['SubKVBucket'] = ref_loads_v['MaxKV'].apply(_bucket_kv)

# Cross-tabulation by count (row-normalized)
cross_count = pd.crosstab(
    ref_loads_v['SubKVBucket'].astype(int),
    ref_loads_v['BusKVBucket'].astype(int),
    normalize='index'
).round(2)

print("\nLoad voltage distribution (by count, row-normalized):")
print(f"{'Sub Max kV':<12} {'-> 69 kV':>10} {'-> 138 kV':>10} {'-> 345 kV':>10}")
print("-"*44)
for sub_kv in sorted(cross_count.index):
    row = cross_count.loc[sub_kv]
    print(f"{sub_kv:<12} {row.get(69, 0):>9.0%} {row.get(138, 0):>9.0%} {row.get(345, 0):>9.0%}")

# Store as dict for use in Cell 38
ref_load_kv_distribution = {}
for sub_kv in cross_count.index:
    ref_load_kv_distribution[sub_kv] = {bus_kv: cross_count.loc[sub_kv].get(bus_kv, 0) for bus_kv in [69, 138, 345]}

# Load size stats by bus kV
print("\nLoad size by voltage level:")
print(f"{'Bus kV':<10} {'Median':>8} {'Mean':>8} {'Max':>8}")
print("-"*38)
for kv in [69, 138, 345]:
    vals = ref_loads_v[ref_loads_v['BusKVBucket'] == kv]['LoadSMW']
    if len(vals) > 0:
        print(f"{kv:<10} {vals.median():>7.1f} {vals.mean():>7.1f} {vals.max():>7.0f}")

# Substation load voltage combination distribution
# For each sub with load, find which kV levels have loads
ref_loads_v['SubNum'] = ref_loads_v['SubNum'].astype(str)
sub_load_combos = ref_loads_v.groupby('SubNum').agg(
    LoadKVs=('BusKVBucket', lambda x: tuple(sorted(set(int(v) for v in x), reverse=True))),
    TotalMW=('LoadSMW', 'sum')
).reset_index()

combo_stats = sub_load_combos.groupby('LoadKVs').agg(
    Subs=('SubNum', 'count'),
    TotalMW=('TotalMW', 'sum')
).sort_index()

total_subs_wl = combo_stats['Subs'].sum()
total_mw_wl = combo_stats['TotalMW'].sum()

print("\nSubstation load voltage combinations:")
print(f"{'Combination':<25} {'Subs':>6} {'% Subs':>7} {'MW':>10} {'% MW':>7}")
print("-"*58)
for combo, row in combo_stats.iterrows():
    label = ' + '.join(f"{kv} kV" for kv in combo)
    pct_s = 100 * row['Subs'] / total_subs_wl
    pct_m = 100 * row['TotalMW'] / total_mw_wl
    print(f"{label:<25} {row['Subs']:>6} {pct_s:>6.1f}% {row['TotalMW']:>10,.0f} {pct_m:>6.1f}%")

# Store combination distribution for Cell 38
# Format: {(345, 138): 0.002, (138, 69): 0.009, ...}
ref_load_combo_distribution = {}
for combo, row in combo_stats.iterrows():
    ref_load_combo_distribution[combo] = row['Subs'] / total_subs_wl

saw.exit()

print(f"\nkV levels per weather zone:")
for wz in sorted(kv_areas):
    print(f"  {wz:20s}: {kv_areas[wz]}")
lap("Reference case data extracted")


# Cell 7: Reference Case Extraction

Opens a PowerWorld `.pwb` via ESA/SAW and extracts:

| Variable | Description |
|----------|-------------|
| `kv_areas` | Sorted kV list per weather zone (filtered to >= 1000 buses) |
| `ref_345_ratio` | Fraction of reference buses at 345 kV → drives EHV target |
| `ref_69_ratio` | Fraction of reference buses at 69 kV |
| `ref_zone_load_shares` | Per-zone load fraction → used for census load scaling |

> **To change:** Update `REFERENCE_CASE` path to your `.pwb` file.


In [ ]:
# ── Create Load Fragments from Census Data ───────────────────────────────────
sgb.load_frags = []
for row in census_records:
    lf = LoadFragment()
    lf.lat = float(row['INTPTLAT'])
    lf.lon = float(row['INTPTLONG'])

    # Filter by ERCOT boundary
    if not is_inside_ercot(lf.lat, lf.lon):
        continue

    # Assign Area (weather zone) and Zone (county)
    lf.area = assign_weather_zone(lf.lat, lf.lon)
    if lf.area is None: continue
    lf.zone = assign_county(lf.lat, lf.lon)

    lf.p = float(row['DP1_0001C']) * 0.0021
    if lf.p < 0.25: continue

    name_parts = str(row['NAME']).split(';')
    tract_num = name_parts[0].split()[-1]
    county_name = name_parts[1].strip().replace(" County, Texas", "").strip()
    lf.name = county_name + "-TX-" + tract_num
    sgb.load_frags.append(lf)

lap(f"Created {len(sgb.load_frags)} ERCOT load fragments")

# Cell 8: Create Load Fragments from Census Data

Converts each Texas census tract into a load fragment. Filters to ERCOT boundary, assigns weather zone and county, estimates load as `population * 0.0021` MW. Produces ~6,200 fragments.


In [ ]:
# ── Create Generator Fragments from EIA 860 Data ─────────────────────────────
import math

def _val(v):
    """Return float from cell value, or 0 if NaN/empty."""
    try:
        f = float(v)
        return 0 if math.isnan(f) else f
    except (TypeError, ValueError):
        return 0

def _str(v):
    """Return stripped string, or '' if NaN."""
    return '' if (v is None or (isinstance(v, float) and math.isnan(v))) else str(v).strip()

plant_map = {}
for plant in eia860_2022_plants:
    if not _val(plant.get("Latitude", 0)): continue
    if _str(plant.get("State", "")) != "TX": continue

    lat = float(plant["Latitude"])
    lon = float(plant["Longitude"])

    if not is_inside_ercot(lat, lon):
        continue

    gf = GenFragment(
        name=_str(plant["Plant Name"]),
        lat=lat, lon=lon,
        plant_code=plant["Plant Code"],
        area=assign_weather_zone(lat, lon),
        units=[]
    )
    if gf.area is None: continue
    gf.zone = assign_county(lat, lon)
    plant_map[plant["Plant Code"]] = gf

for gen in eia860_2022_gens_operable:
    if gen["Plant Code"] not in plant_map: continue
    gf = plant_map[gen["Plant Code"]]
    tech = _str(gen.get("Technology", ""))
    pmax = _val(gen.get("Summer Capacity (MW)")) or _val(gen.get("Nameplate Capacity (MW)"))
    pmin = _val(gen.get("Minimum Load (MW)", 0))

    # Skip generators commissioned >= 2024 or retired before 2024
    curr_year = _val(gen.get("Current Year", 0))
    ret_year  = _val(gen.get("Planned Retirement Year", 0))
    if curr_year and curr_year >= 2024: continue
    if ret_year and ret_year < 2024: continue

    genunit = GenUnit(fueltype=tech, pmax=pmax, pmin=pmin,
                      unit_id=_str(gen.get("Generator ID", "")))
    gf.units.append(genunit)

sgb.gen_frags = [g for g in plant_map.values()
                 if len(g.units) > 0 and sum(gu.pmax for gu in g.units) > 5]

lap(f"Created {len(sgb.gen_frags)} ERCOT gen fragments")
print(f"Total gen units: {sum(len(g.units) for g in sgb.gen_frags)}")
gen_cap_total = sum(sum(gu.pmax for gu in g.units) for g in sgb.gen_frags)
print(f"Total gen capacity: {gen_cap_total:.0f} MW")

In [ ]:
# ── Insert GIS Planned Generators (ERCOT GIS Report, IA Signed) ──────────────
# Loads GIS generator data from 4 input files, merges them, filters to approved
# generators (IA Signed), and converts to GenFragment/GenUnit objects.
import glob as _glob

# Load GIS generators (EIA 860 format, 2026 January vintage)
gis_gen = pd.read_csv(os.path.join(sgb.input_folder, 'GIS_generators_2026_January.csv'), encoding='latin1')
# Load plant coordinates (made-up geographic coordinates per plant)
gis_plant = pd.read_csv(os.path.join(sgb.input_folder, 'GIS_Plant_madeup_updated.csv'), encoding='latin1')
# Load ERCOT GIS Report (Large Gen header=30, Small Gen header=14)
_xlsx_files = _glob.glob(os.path.join(sgb.input_folder, 'RPT*.xlsx'))
_xlsx_path = _xlsx_files[0]
ercot_large = pd.read_excel(_xlsx_path, sheet_name='Project Details - Large Gen', header=30)
ercot_small = pd.read_excel(_xlsx_path, sheet_name='Project Details - Small Gen', header=14)
ercot_gis = pd.concat([ercot_large, ercot_small], ignore_index=True)
# Load fuel type key
key_gen = pd.read_csv(os.path.join(sgb.input_folder, 'key_gen_type.csv'), encoding='latin1')

print(f"GIS generators: {len(gis_gen)} | Plants: {len(gis_plant)} | ERCOT GIS: {len(ercot_gis)} | Key: {len(key_gen)}")

# ── Merge generators + plant coordinates on Utility ID ────────────────────────
gis_df = gis_gen.merge(
    gis_plant[['Utility ID', 'Latitude', 'Longitude']],
    on='Utility ID', how='left', suffixes=('', '_plant')
)

# ── Map fuel types via key_gen_type ───────────────────────────────────────────
_fuel_map = key_gen.set_index(['Prime Mover', 'Energy Source 1'])['Technology']
gis_df['Technology_Mapped'] = gis_df.set_index(['Prime Mover', 'Energy Source 1']).index.map(
    lambda x: _fuel_map.get(x, 'Other')
)

# ── Merge with ERCOT GIS Report for IA Signed + CDR zone ─────────────────────
_ercot_keep = ['INR', 'IA Signed', 'CDR Reporting Zone', 'Capacity (MW)', 'Fuel', 'Technology']
_ercot_keep = [c for c in _ercot_keep if c in ercot_gis.columns]
_ercot_sub = ercot_gis[_ercot_keep].copy()
_ercot_sub['INR'] = _ercot_sub['INR'].astype(str).str.strip()
gis_df['Utility ID'] = gis_df['Utility ID'].astype(str).str.strip()
_ercot_sub = _ercot_sub.drop_duplicates(subset='INR', keep='first')
_ercot_sub = _ercot_sub[_ercot_sub['INR'].notna() & (_ercot_sub['INR'] != 'nan')]
_ercot_sub = _ercot_sub.rename(columns={'INR': 'Utility ID'})
gis_df = gis_df.merge(_ercot_sub, on='Utility ID', how='left', suffixes=('', '_ercot'))

# ── Filter to approved generators (IA Signed not null) ────────────────────────
gis_approved = gis_df[gis_df['IA Signed'].notna()].copy()
gis_approved['Nameplate Capacity (MW)'] = pd.to_numeric(gis_approved['Nameplate Capacity (MW)'], errors='coerce')
print(f"Approved generators (IA Signed): {len(gis_approved)}")
print(f"  Total capacity: {gis_approved['Nameplate Capacity (MW)'].sum():,.0f} MW")
print(f"  By technology: {gis_approved['Technology_Mapped'].value_counts().to_dict()}")

# ── Convert to GenFragments ───────────────────────────────────────────────────
existing_plant_codes = set(plant_map.keys())
n_gis_frags = 0
n_gis_skipped = 0

for pc, group in gis_approved.groupby('Plant Code'):
    if pc in existing_plant_codes:
        n_gis_skipped += 1
        continue

    row0 = group.iloc[0]
    lat = row0.get('Latitude', np.nan)
    lon = row0.get('Longitude', np.nan)

    if pd.isna(lat) or pd.isna(lon):
        continue
    if not is_inside_ercot(float(lat), float(lon)):
        continue

    area = assign_weather_zone(float(lat), float(lon))
    if area is None:
        continue

    gf = GenFragment(
        name=_str(row0.get('Plant Name', f'GIS_{pc}')),
        lat=float(lat), lon=float(lon),
        plant_code=str(pc),
        area=area,
        units=[]
    )
    gf.zone = assign_county(float(lat), float(lon))

    for _, row in group.iterrows():
        pmax = _val(row.get('Nameplate Capacity (MW)', 0))
        tech = _str(row.get('Technology_Mapped', '')) 
        if not tech:
            tech = _str(row.get('Technology', ''))
        gu = GenUnit(fueltype=tech, pmax=pmax, pmin=0,
                     unit_id=_str(row.get('Generator ID', '')))
        gf.units.append(gu)

    if len(gf.units) > 0 and sum(gu.pmax for gu in gf.units) > 0:
        sgb.gen_frags.append(gf)
        n_gis_frags += 1

print(f"\nAdded {n_gis_frags} GIS planned gen fragments (skipped {n_gis_skipped} already operable)")
print(f"Total gen fragments: {len(sgb.gen_frags)}")
gis_gen_cap = sum(sum(gu.pmax for gu in g.units) for g in sgb.gen_frags)
print(f"Total gen capacity (operable + GIS planned): {gis_gen_cap:,.0f} MW")
lap("Inserted GIS planned generators")

# ── Combined Generation by Fuel Type (2031 Case) ─────────────────────────────
# Maps all gen_frags (EIA operable + GIS planned) to standardized fuel categories
_fuel_category_map = {
    'Solar Photovoltaic': 'Solar',
    'Batteries': 'Battery Storage',
    'Onshore Wind Turbine': 'Wind',
    'Natural Gas Fired Combined Cycle': 'Gas, Combined Cycle',
    'Natural Gas Steam Turbine': 'Gas, Steam Turbine',
    'Natural Gas Internal Combustion Engine': 'Gas Turbine',
    'Natural Gas Fired Combustion Turbine': 'Gas Turbine',
    'Nuclear': 'Nuclear',
    'Petroleum Liquids': 'Diesel',
    'Conventional Hydroelectric': 'Hydro',
    'Landfill Gas': 'Biomass',
    'Wood/Wood Waste Biomass': 'Biomass',
    'Municipal Solid Waste': 'Biomass',
    'Other Waste Biomass': 'Biomass',
    'Geothermal': 'Other',
    'All Other': 'Other',
    'Other Gases': 'Other',
    'Other Natural Gas': 'Gas Turbine',
    'Coal': 'Coal',
    'Hydroelectric Pumped Storage': 'Hydro',
    'Solar Thermal without Energy Storage': 'Solar',
    'Solar Thermal with Energy Storage': 'Solar',
    'Flywheels': 'Battery Storage',
    'Conventional Steam Coal': 'Coal',
    'Petroleum Coke': 'Coal',
}

from collections import defaultdict
fuel_summary = defaultdict(lambda: {'units': 0, 'mw': 0.0})
for gf in sgb.gen_frags:
    for gu in gf.units:
        cat = _fuel_category_map.get(gu.fueltype, 'Other')
        fuel_summary[cat]['units'] += 1
        fuel_summary[cat]['mw'] += gu.pmax

# Display order matching user's reference table
_display_order = ['Biomass', 'Coal', 'Diesel', 'Gas, Combined Cycle',
                  'Gas Turbine', 'Gas, Steam Turbine', 'Hydro', 'Nuclear',
                  'Solar', 'Battery Storage', 'Wind', 'Other']

print("\n" + "="*60)
print("Combined Generation by Fuel Type (2031 Synthetic Case)")
print("="*60)
print(f"{'Fuel':<25} {'Units':>6} {'Capacity (MW)':>15}")
print("-"*50)
total_units = 0
total_mw = 0.0
for fuel in _display_order:
    d = fuel_summary.get(fuel, {'units': 0, 'mw': 0.0})
    if d['units'] > 0:
        print(f"{fuel:<25} {d['units']:>6} {d['mw']:>15,.0f}")
        total_units += d['units']
        total_mw += d['mw']
print("-"*50)
print(f"{'TOTAL':<25} {total_units:>6} {total_mw:>15,.0f}")


# Cells 9a–9b: Create Generator Fragments

**9a (EIA 860):** Converts operable plants/generators to gen fragments (~600 plants, ~148 GW).

**9b (GIS Queue):** Adds planned generators from ERCOT GIS Report (IA Signed only, ~565 units, ~120 GW). Skips plants already in EIA 860.

**Combined fuel type table** is printed at the end of Cell 9b. It maps EIA/GIS `Technology` values to summary categories:

| EIA / GIS Technology | Summary Category |
|---------------------|-----------------|
| Solar Photovoltaic | Solar |
| Batteries, Flywheels | Battery Storage |
| Onshore Wind Turbine | Wind |
| Natural Gas Fired Combined Cycle | Gas, Combined Cycle |
| Natural Gas Steam Turbine | Gas, Steam Turbine |
| Natural Gas Fired Combustion Turbine, Natural Gas Internal Combustion Engine | Gas Turbine |
| Nuclear | Nuclear |
| Conventional Steam Coal, Petroleum Coke | Coal |
| Petroleum Liquids | Diesel |
| Conventional Hydroelectric, Hydroelectric Pumped Storage | Hydro |
| Landfill Gas, Wood/Wood Waste Biomass, Municipal Solid Waste | Biomass |
| Geothermal, All Other, Other Gases | Other |


In [ ]:
# ── Synthetic Large Load Allocation ──────────────────────────────────────────
# Force reload in case module was updated on disk
import importlib
import large_load_allocation
importlib.reload(large_load_allocation)
from large_load_allocation import run_large_load_allocation, LARGE_LOAD_MW, plot_spatial_validation

# Re-open reference case for large load extraction
saw_ll = SAW(REFERENCE_CASE)

# ── Tunable: number of synthetic large loads ─────────────────────────────────
# Increase N_LARGE_LOADS to add more loads at benchmark MW sizes (no MW scaling).
# Each load keeps its realistic per-zone lognormal MW from the reference case.
# Default ~250 produces ~52 GW total at ~208 MW avg per load.
N_LARGE_LOADS = 250

synthetic_large_loads, ref_large_loads_gdf = run_large_load_allocation(
    saw=saw_ll,
    weather_zone_gdf=ercot_zones_gdf,          # from shapefile cell
    boundary_polygon=ercot_boundary,            # merged ERCOT polygon
    fit_thomas=True,                            # set False to skip fitting (debug)
    sigma_fallback_km=50.0,                     # used only when fit_thomas=False
    zone_col='ERCOT_Clim',
    seed=42,
    n_loads=N_LARGE_LOADS,                      # target number of large loads
)

saw_ll.exit()
lap(f"Synthetic large loads generated: {len(synthetic_large_loads)}")

# Cell 10: Synthetic Large Load Allocation

Runs `large_load_allocation.py` — generates synthetic large loads (>= 75 MW) matching the spatial pattern of the reference case.

## Tunable Parameters

| Variable | Default | Effect |
|----------|---------|--------|
| `N_LARGE_LOADS` | 250 | Target number of synthetic large loads. Increase to add more loads at benchmark MW sizes. |
| `seed` | 42 | Random seed for reproducibility. |

---

In [ ]:
plot_spatial_validation(
    ref_loads_df      = ref_large_loads_gdf,
    synthetic_gdf     = synthetic_large_loads,
    boundary_polygon  = ercot_boundary,
    weather_zone_gdf  = ercot_zones_gdf,
    zone_col          = 'ERCOT_Clim',
    r_max_km          = 200,
    n_r               = 80,
    seed              = 42,
)

# Cell 11: Validation Plot

3-panel figure: Ripley's L(r), nearest-neighbor CDF, and spatial map comparing synthetic vs reference vs Poisson baseline.


In [ ]:
# Pre-processing
sgb.assign_load_q()
for g in sgb.gen_frags:
    for gu in g.units: gu.sbase = round(gu.pmax/.9, 1)
sgb.convert_gen_fuel()
sgb.assign_gen_qlims()
sgb.assign_gen_cost()

# Cell 12: Pre-Processing

Assigns reactive power (Q), generator MVA base, fuel type mapping, Q limits, and cost curves.


In [ ]:
# ── Create Areas (Weather Zones) and Zones (Counties) ─────────────────────────
sgb.create_areas()
sgb.create_zones()
lap("Areas and Zones created")

# Cell 13: Create Areas and Zones

Creates Area (weather zone) and Zone (county) objects from the assignments already set on each fragment.


In [ ]:
# ── Clustering into Substations ───────────────────────────────────────────────
# Target ~4,200 load-based subs for ~5,000 total subs. After gen-only subs, core total ≈ 4,700+.
# Large-load subs (~456) and switching subs (TBD in transmission planning)
# are added separately and NOT counted toward this core target.
#
# Reference case (Texas 2k) active sub ratios for comparison:
#   Load only: 74.7%  |  Gen only: 2.4%  |  Both: 22.9%

import importlib, sgb_suite.sub_planning
importlib.reload(sgb_suite.sub_planning)

sgb.cluster_load_frags(5000)
sgb.create_subs_old()

# Post-processing
for sub in sgb.syn_subs: sub.kvs = [100]

# ── Co-locate gen-only subs with nearby load subs ─────────────────────────────
# create_subs_old() uses exact (lat,lon) matching, so gens never share a sub
# with loads. Fix: merge gen-only subs into the nearest load-bearing sub
# within a distance threshold, matching the ~22.9% "both" ratio in reference.
from math import sqrt

CO_LOCATE_THRESHOLD_DEG = 0.15  # ~17 km — gen plants within this join a load sub

gen_only_subs = [s for s in sgb.syn_subs if s.gen_frags and not s.load_frags]
load_subs = [s for s in sgb.syn_subs if s.load_frags]

n_merged = 0
subs_to_remove = []
for gs in gen_only_subs:
    # Find nearest load sub
    best_dist = float('inf')
    best_ls = None
    for ls in load_subs:
        d = sqrt((gs.lat - ls.lat)**2 + (gs.lon - ls.lon)**2)
        if d < best_dist:
            best_dist = d
            best_ls = ls
    if best_ls is not None and best_dist <= CO_LOCATE_THRESHOLD_DEG:
        # Move gen frags to load sub
        best_ls.gen_frags.extend(gs.gen_frags)
        subs_to_remove.append(gs)
        n_merged += 1

# Remove merged gen-only subs
for s in subs_to_remove:
    sgb.syn_subs.remove(s)

print(f"Co-located {n_merged} gen-only subs with nearby load subs (threshold: {CO_LOCATE_THRESHOLD_DEG} deg)")

# Create power flow case with just substations
sgb.create_case_old()

n_load_only = sum(1 for s in sgb.syn_subs if s.load_frags and not s.gen_frags)
n_gen_only = sum(1 for s in sgb.syn_subs if s.gen_frags and not s.load_frags)
n_both = sum(1 for s in sgb.syn_subs if s.load_frags and s.gen_frags)
n_active = len(sgb.syn_subs)
print(f"\nCore substations: {n_active}")
print(f"  Load only: {n_load_only} ({100*n_load_only/n_active:.1f}%)  [ref: 74.7%]")
print(f"  Gen only:  {n_gen_only} ({100*n_gen_only/n_active:.1f}%)  [ref: 2.4%]")
print(f"  Both:      {n_both} ({100*n_both/n_active:.1f}%)  [ref: 22.9%]")
lap("Substation clustering done")

# Cell 14: Clustering into Substations

`cluster_load_frags(4200)` merges ~6,200 load fragments into ~4,200 substations. Then `create_subs_old()` adds ~500 gen-only subs for ~4,700 core substations.

> **Tuning:** Change the argument to `cluster_load_frags(N)` to adjust core substation count.


In [ ]:
# ── EHV Clustering + kV Assignment ────────────────────────────────────────────
# See markdown cell above for tunable parameters and pipeline description.

# ── Tunable Parameters ───────────────────────────────────────────────────────
FINAL_BUS_TARGET = 10000  # approximate total bus count (after Delaunay propagation)

subs = sgb.wb.subs
sub_coords = [(s.longitude, s.latitude) for s in subs]

# Target 345 kV bus count from reference case ratio
# Each EHV sub = 1 bus at 345 kV, so target_n_ehv = target 345 kV buses
ehv_target_ratio = ref_345_ratio if 'ref_345_ratio' in dir() else 0.154
target_345_buses = int(round(ehv_target_ratio * FINAL_BUS_TARGET))
target_n_ehv = target_345_buses

# nc controls how many geographic clusters KMeans creates
# More clusters = finer geographic spread of EHV subs
# Original do_sub_planning.py used nc=400 for ~70k bus EI case
nc = max(50, target_n_ehv // 5)  # ~5 EHV subs per cluster on average
print(f"EHV target: {target_n_ehv} subs ({ehv_target_ratio:.3f} ratio), KMeans nc={nc}")

class EHV_Cluster:
    def __init__(self):
        self.subs = []

clusters = [EHV_Cluster() for _ in range(nc)]
km = KMeans(n_clusters=nc, n_init=10).fit(sub_coords)
lap("KMeans clustering done")

for s, c in zip(subs, km.labels_):
    s.is_ehv = False
    s.cluster = c
    clusters[c].subs.append(s)

# Assign EHV identity based on cluster power content (from original)
for c in clusters:
    gen_p = sum(g.pmax for s in c.subs for g in s.gens)
    load_p = sum(l.p for s in c.subs for l in s.loads)
    p = gen_p + load_p
    c.n_ehv = min(10, int(round(p / 1500, 0)), len(c.subs))
    if p < 1200: c.n_ehv = 0
    c.subs.sort(key=lambda s: sum(-g.pmax for g in s.gens))
    i_ehv = 0
    for sub in c.subs:
        if i_ehv == c.n_ehv: break
        if sum(g.pmax for g in sub.gens) < 300: break
        i_ehv += 1
        sub.is_ehv = True
    while i_ehv < c.n_ehv:
        s = c.subs[int(random() * len(c.subs))]
        if s.is_ehv: continue
        i_ehv += 1
        s.is_ehv = True

# Adjust EHV count to match reference ratio (±20% tolerance)
current_ehv = [s for s in subs if s.is_ehv]
tolerance = 0.20
ehv_low = int(target_n_ehv * (1 - tolerance))
ehv_high = int(target_n_ehv * (1 + tolerance))

if len(current_ehv) > ehv_high:
    current_ehv.sort(key=lambda s: sum(g.pmax for g in s.gens) + sum(l.p for l in s.loads), reverse=True)
    for s in current_ehv[target_n_ehv:]:
        s.is_ehv = False
    print(f"Trimmed EHV: {len(current_ehv)} -> {target_n_ehv} (above {ehv_high})")
elif len(current_ehv) < ehv_low:
    all_by_power = sorted(subs, key=lambda s: sum(g.pmax for g in s.gens) + sum(l.p for l in s.loads), reverse=True)
    promoted = 0
    for s in all_by_power:
        if len(current_ehv) + promoted >= target_n_ehv: break
        if not s.is_ehv:
            s.is_ehv = True
            promoted += 1
    print(f"Promoted {promoted} subs to EHV: {len(current_ehv)} -> {len(current_ehv)+promoted} (below {ehv_low})")
else:
    print(f"EHV count {len(current_ehv)} within [{ehv_low}, {ehv_high}] — no adjustment")

# ── Assign kV levels ─────────────────────────────────────────────────────────
for s in subs:
    area_name = s.area.name
    kvs = kv_areas.get(area_name, [69, 138, 345])
    base = kvs[1] if len(kvs) >= 2 else kvs[0]
    s.kv_levels = [base]
    if s.is_ehv:
        s.kv_levels.append(kvs[-1])

# ── Reassign smallest-load subs to 69 kV base ─────────────────────────────
TARGET_69_RATIO = 0.15    # target fraction of 69 kV buses (reference: ~15.3%)
current_bus_count_pre = sum(len(s.kv_levels) for s in subs)
estimated_extra = max(0, FINAL_BUS_TARGET - current_bus_count_pre)
estimated_final = current_bus_count_pre + estimated_extra
target_69_buses = int(round(TARGET_69_RATIO * estimated_final))

current_69 = sum(1 for s in subs for kv in s.kv_levels if kv == 69)
n_69_needed = target_69_buses - current_69

if n_69_needed > 0:
    candidates_69 = [s for s in subs if not s.is_ehv and s.kv_levels == [138]]
    candidates_69.sort(key=lambda s: sum(l.p for l in s.loads))
    n_reassigned = 0
    for s in candidates_69:
        if n_reassigned >= n_69_needed:
            break
        s.kv_levels = [69]
        n_reassigned += 1
    print(f"Reassigned {n_reassigned} subs from 138 kV to 69 kV (target: {target_69_buses} of ~{estimated_final} buses)")

pre_delaunay_buses = sum(len(s.kv_levels) for s in subs)
print(f"Pre-Delaunay bus count: {pre_delaunay_buses}")

# Cross-area kV propagation via Delaunay neighbors (from original)
gg = geograph.GeoGraph(subs)
gg.Delaunay(1)
for s in subs:
    for s2 in gg.g.neighbors(s):
        kv2 = s2.kv_levels[0]
        if kv2 in s.kv_levels: continue
        if kv2 >= s.kv_levels[0]: continue
        if random() < 0.3:
            s.kv_levels.append(kv2)

ehv_subs = [s for s in subs if max(s.kv_levels) > 300]
gg = geograph.GeoGraph(ehv_subs)
gg.Delaunay(1)
for u, v, a in list(gg.g.edges(data=True)):
    if a["dist"] > 250:
        gg.g.remove_edge(u, v)
for s in ehv_subs:
    for s2 in gg.g.neighbors(s):
        kv2 = max(s2.kv_levels)
        if kv2 in s.kv_levels: continue
        if kv2 >= max(s.kv_levels): continue
        if random() < 0.4:
            s.kv_levels.append(kv2)

# ── 765 kV Substation Selection (ERCOT Long-Range Plan) ──────────────────────
# Select existing 345 kV subs near approximate corridor waypoints and upgrade
# to 765 kV. Waypoints are offset from real ERCOT locations to avoid replicating
# the actual grid.
from math import radians, sin, cos, sqrt, atan2

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371
    dlat, dlon = radians(lat2 - lat1), radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

# Approximate waypoints (offset ~20-50 km from real ERCOT 765 kV nodes)
# West cluster: 3 subs near NM border
PERMIAN_WEST = [
    (31.8, -106.2),   # Far west near NM border
    (32.2, -104.2),   # West TX north
    (30.5, -104.0),   # West TX south
]
# Middle corridor: 6 subs spread east from Permian Basin to central TX
PERMIAN_MIDDLE = [
    (33.0, -100.5),   # North-central TX (west)
    (31.8, -101.0),   # Central TX west
    (30.5, -98.5),    # Central TX south
    (32.3, -97.5),    # Central TX north (near DFW)
    (29.5, -99.0),    # South TX
]
PERMIAN_WAYPOINTS = PERMIAN_WEST + PERMIAN_MIDDLE

# Eastern route: 4 subs from central TX east and south
EASTERN_WAYPOINTS = [
    (32.3, -95.2),    # East of DFW
    (31.3, -94.8),    # East TX
    (29.7, -95.8),    # Southeast TX / Gulf
    (28.0, -97.2),    # South TX coast
]

ehv_candidates = [s for s in subs if s.is_ehv]
selected_765 = set()
route_assignments = []  # (sub, route_name, waypoint_idx, dist_km)

for route_name, waypoints in [("Permian West", PERMIAN_WEST),
                               ("Permian Middle", PERMIAN_MIDDLE),
                               ("Eastern", EASTERN_WAYPOINTS)]:
    for wp_idx, (wp_lat, wp_lon) in enumerate(waypoints):
        best_sub, best_dist = None, float('inf')
        for s in ehv_candidates:
            if id(s) in selected_765:
                continue
            d = haversine_km(wp_lat, wp_lon, s.latitude, s.longitude)
            if d < best_dist:
                best_dist = d
                best_sub = s
        if best_sub is not None:
            best_sub.kv_levels.append(765)
            selected_765.add(id(best_sub))
            route_assignments.append((best_sub, route_name, wp_idx, best_dist))

print(f"── 765 kV Substation Selection ──")
print(f"  Total 765 kV subs: {len(selected_765)}")
for s, route, idx, dist in route_assignments:
    print(f"  [{route:14s} #{idx+1}] {s.name:30s}  ({s.latitude:.2f}, {s.longitude:.2f})  dist={dist:.0f} km")


# Bus count after Delaunay
current_bus_count = sum(len(s.kv_levels) for s in subs)
print(f"Bus count after Delaunay: {current_bus_count}")

# Add parallel buses (same kV, split load) if under target
if current_bus_count < FINAL_BUS_TARGET:
    n_needed = (FINAL_BUS_TARGET - current_bus_count) // 2  # only fill half the gap
    candidates = sorted(
        [s for s in subs if len(s.kv_levels) == 1 and sum(l.p for l in s.loads) > 1],
        key=lambda s: sum(l.p for l in s.loads), reverse=True
    )
    n_added = 0
    for s in candidates:
        if n_added >= n_needed:
            break
        s.kv_levels.append(s.kv_levels[0])
        s._split_load = True
        n_added += 1
    print(f"Added {n_added} parallel buses (target ~{FINAL_BUS_TARGET})")

print(f"Final bus count: {sum(len(s.kv_levels) for s in subs)}")

# ── Summary ──────────────────────────────────────────────────────────────────
n_ehv = sum(1 for s in subs if s.is_ehv)
total_buses_est = sum(len(s.kv_levels) for s in subs)

from collections import Counter
kv_counter = Counter(kv for s in subs for kv in s.kv_levels)

print(f"\nEHV subs: {n_ehv}/{len(subs)} ({100*n_ehv/len(subs):.1f}%)")
print(f"Estimated bus count: {total_buses_est}")
print(f"\nBus count by kV level:")
for kv in sorted(kv_counter):
    print(f"  {kv:>6} kV: {kv_counter[kv]:>5} ({100*kv_counter[kv]/total_buses_est:.1f}%)")

if 'ref_kv_counts' in dir():
    print(f"\nRef case comparison:")
    for kv in sorted(set(list(kv_counter.keys()) + list(ref_kv_counts.index))):
        syn = kv_counter.get(kv, 0)
        ref = ref_kv_counts.get(kv, 0)
        print(f"  {kv:>6} kV: syn {syn:>5} ({100*syn/total_buses_est:.1f}%) vs ref {ref:>5} ({100*ref/ref_total_buses:.1f}%)")

# NOTE: Switching substations will be added during transmission planning.

lap("EHV clustering + kV assignment done")




# Cell 15: EHV Clustering + kV Assignment

Identifies EHV substations, assigns voltage levels, selects 765 kV corridor subs, and adds parallel buses.

## Tunable Parameters

| Variable | Default | Effect |
|----------|---------|--------|
| `FINAL_BUS_TARGET` | 10,000 | Approximate total bus count. Parallel buses fill half the gap to this target. |
| `ehv_target_ratio` | ~0.154 | Fraction of buses at 345 kV. Loaded from reference case. |
| `TARGET_69_RATIO` | 0.15 | Fraction of buses at 69 kV. Reassigns smallest-load subs. |
| Delaunay non-EHV | 0.30 | 30% chance of adding step-down bus from Delaunay neighbor. |
| Delaunay EHV | 0.40 | 40% chance for EHV subs. |
| `tolerance` | 0.20 | EHV count can deviate ±20% before adjustment. |

## Pipeline Steps

1. **KMeans clustering** — groups subs into geographic clusters
2. **EHV assignment** — each cluster assigns EHV subs based on generation + load MW
3. **EHV adjustment** — promotes or trims subs to match target ratio
4. **Base kV assignment** — all subs get base kV from their area; EHV subs get 345 kV
5. **69 kV reassignment** — smallest-load non-EHV subs get 69 kV
6. **Delaunay step-down** — neighboring subs exchange lower kV levels (30% non-EHV, 40% EHV)
7. **765 kV selection** — nearest EHV subs to corridor waypoints get 765 kV
8. **Parallel buses** — adds duplicate-kV buses to largest-load subs to reach bus target (fills half the gap)


# Cell 15b: 765 kV Corridor and Weather Zone Visualization


In [ ]:
# ── 765 kV Corridor Visualization ────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(14, 10))

# Plot ERCOT boundary
if 'ercot_zones_gdf' in dir():
    ercot_zones_gdf.boundary.plot(ax=ax, color='lightgray', linewidth=0.5)

# All subs as light gray dots
all_lats = [s.latitude for s in subs]
all_lons = [s.longitude for s in subs]
ax.scatter(all_lons, all_lats, s=1, c='lightgray', alpha=0.3, label='All subs')

# 345 kV subs as small blue dots
ehv_lats = [s.latitude for s in subs if s.is_ehv and 765 not in s.kv_levels]
ehv_lons = [s.longitude for s in subs if s.is_ehv and 765 not in s.kv_levels]
ax.scatter(ehv_lons, ehv_lats, s=8, c='steelblue', alpha=0.4, label='345 kV subs')

# 765 kV subs by route (markers only, no connecting lines)
for s, route, idx, d in route_assignments:
    if "Permian" in route:
        color, marker_label = 'darkorange', '765 kV Permian'
    else:
        color, marker_label = 'hotpink', '765 kV Eastern'
    ax.scatter(s.longitude, s.latitude, s=200, c=color, marker='*', zorder=5,
               edgecolors='black', linewidths=0.5)
    label_txt = f'PW{idx+1}' if "West" in route else f'PM{idx+1}' if "Middle" in route else f'E{idx+1}'
    ax.annotate(label_txt, (s.longitude, s.latitude), fontsize=7,
                xytext=(5, 5), textcoords='offset points')

# Waypoint targets as small x markers
for wp_lat, wp_lon in PERMIAN_WAYPOINTS:
    ax.plot(wp_lon, wp_lat, 'x', c='darkorange', markersize=6, alpha=0.5)
for wp_lat, wp_lon in EASTERN_WAYPOINTS:
    ax.plot(wp_lon, wp_lat, 'x', c='hotpink', markersize=6, alpha=0.5)

# Manual legend entries (avoid duplicates)
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='lightgray', markersize=4, label='All subs'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='steelblue', markersize=6, label='345 kV subs'),
    Line2D([0],[0], marker='*', color='w', markerfacecolor='darkorange', markersize=12, label='765 kV Permian'),
    Line2D([0],[0], marker='*', color='w', markerfacecolor='hotpink', markersize=12, label='765 kV Eastern'),
    Line2D([0],[0], marker='x', color='darkorange', markersize=6, linestyle='None', label='Target waypoints'),
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=9)

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('765 kV Substation Selection — Synthetic ERCOT Grid')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()



In [ ]:

# ── Weather Zone and County Visualization ────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(20, 9))

# ── Panel 1: Weather Zones ──────────────────────────────────────────────────
ax = axes[0]
zone_colors = {
    'COAST': '#1f77b4', 'EAST': '#2ca02c', 'FAR WEST': '#d62728',
    'NORTH': '#9467bd', 'NORTH CENTRAL': '#ff7f0e', 'SOUTH': '#8c564b',
    'SOUTH CENTRAL': '#e377c2', 'WEST': '#17becf'
}

# Plot each zone individually to guarantee correct color
for _, row in ercot_zones_gdf.iterrows():
    zone_name = row['ERCOT_Clim']
    color = zone_colors.get(zone_name, 'gray')
    import geopandas as gpd
    gpd.GeoDataFrame([row], geometry='geometry').plot(
        ax=ax, color=color, edgecolor='black', linewidth=0.8)

# Add zone labels at centroids
for _, row in ercot_zones_gdf.iterrows():
    centroid = row['geometry'].centroid
    ax.text(centroid.x, centroid.y, row['ERCOT_Clim'].title(), fontsize=9,
            fontweight='bold', ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))

patches = [mpatches.Patch(facecolor=c, edgecolor='black', label=z)
           for z, c in zone_colors.items()]
ax.legend(handles=patches, loc='lower left', fontsize=8, framealpha=0.9)
ax.set_title('ERCOT Weather Zones', fontsize=14, fontweight='bold')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude'); ax.set_aspect('equal')

# ── Panel 2: Counties (colored by weather zone) ─────────────────────────────
ax = axes[1]
import geopandas as gpd
# Assign each county a light color based on which weather zone it falls in
county_copy = county_gdf.copy()
county_copy['zone'] = None
for idx, row in county_copy.iterrows():
    centroid = row['geometry'].centroid
    for _, zrow in ercot_zones_gdf.iterrows():
        if zrow['geometry'].contains(centroid):
            county_copy.at[idx, 'zone'] = zrow['ERCOT_Clim']
            break

zone_light_colors = {
    'COAST': '#aed6f1', 'EAST': '#abebc6', 'FAR WEST': '#f5b7b1',
    'NORTH': '#d2b4de', 'NORTH CENTRAL': '#fad7a0', 'SOUTH': '#d7ccc8',
    'SOUTH CENTRAL': '#f5b7db', 'WEST': '#aee8e8'
}
county_copy['fill_color'] = county_copy['zone'].map(zone_light_colors).fillna('#e8e8e8')

for _, row in county_copy.iterrows():
    gpd.GeoDataFrame([row], geometry='geometry').plot(
        ax=ax, color=row['fill_color'], edgecolor='gray', linewidth=0.3)

ercot_zones_gdf.boundary.plot(ax=ax, color='black', linewidth=1.5)

county_counts = {}
for s in subs:
    c = getattr(s, 'county', None)
    if c:
        county_counts[c] = county_counts.get(c, 0) + 1

top_counties = sorted(county_counts.items(), key=lambda x: -x[1])[:20]
top_names = {c for c, _ in top_counties}
for _, row in county_gdf.iterrows():
    name = row.get('US_Count_1', '').replace(' County', '').strip()
    if name in top_names:
        centroid = row['geometry'].centroid
        ax.text(centroid.x, centroid.y, name, fontsize=6, ha='center', va='center',
                bbox=dict(boxstyle='round,pad=0.1', facecolor='white', alpha=0.6))

ax.set_title(f'Texas Counties ({len(county_gdf)} total, {len(county_counts)} with subs)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude'); ax.set_aspect('equal')

plt.tight_layout()
plt.savefig('weather_zones_counties.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: weather_zones_counties.png')





In [ ]:
# ── Bus Creation + Gen Voltage Assignment ─────────────────────────────────────
busnum = len(sgb.wb.buses) + 1
for s in subs:
    b0 = s.buses[0]
    b0.nominal_kv = s.kv_levels[0]
    b0.name = s.name + f"_{b0.nominal_kv}"
    kv_seen = {s.kv_levels[0]: 1}
    for kv in s.kv_levels[1:]:
        b = Bus(s, busnum)
        busnum += 1
        b.nominal_kv = kv
        kv_seen[kv] = kv_seen.get(kv, 0) + 1
        if kv_seen[kv] > 1:
            b.name = s.name + f"_{kv}_{kv_seen[kv]}"
        else:
            b.name = s.name + f"_{kv}"
        b.zone_number = b0.zone_number

# Move large generators to highest voltage bus
for s in subs:
    behv = max(s.buses, key=lambda b: b.nominal_kv)
    if behv.nominal_kv < 300: continue
    gens = list(s.gens)
    gens.sort(key=lambda g: g.pmax, reverse=True)  # largest first
    total_g = 0
    for g in gens:
        if g.pmax >= 75:  # move gens >= 75 MW to EHV bus
            total_g += g.pmax
            if len(behv.nodes) == 0:
                n = Node(behv, behv.number)
                n.name = behv.name
            else:
                n = behv.nodes[0]
            g.node = n

# ── Load Voltage Allocation (Benchmark-Based) ─────────────────────────────
# From reference case statistics:
#   Bus kV   Median MW   Mean MW   Max MW
#   69         3.2         5.6      232     Small distribution loads
#   138       14.8        19.3      850     Bulk system loads
#   345       20.0       118.2    1,400     Large industrial / data center
#
# Rules:
# 1. Loads at 345 kV subs: only place on 345 kV bus if MW >= MIN_345_LOAD_MW,
#    otherwise place on the lowest existing bus (typically 138 kV)
# 2. High-MW 345 kV subs (top 16.7%) get split: 64% MW at 345, 36% at 138
# 3. High-MW 138 kV subs (top 1.2%) get split: 97% MW at 138, 3% at 69
# 4. All others: single load at the sub's primary voltage level
# 5. NEVER create new buses here — only use existing buses from Cell 15

def _bucket_kv(kv):
    if kv <= 69: return 69
    elif kv <= 200: return 138
    else: return 345

def _find_bus(s, target_kv):
    """Find an existing bus at target_kv bucket. If none, return lowest-kV bus."""
    for b in s.buses:
        if _bucket_kv(b.nominal_kv) == target_kv:
            return b
    # Fallback: return bus with nearest lower kV, or first bus
    sorted_buses = sorted(s.buses, key=lambda b: b.nominal_kv)
    return sorted_buses[0]

def _ensure_node(bus):
    """Ensure bus has at least one node, return it."""
    if len(bus.nodes) == 0:
        nd = Node(bus, bus.number)
        nd.name = bus.name
    return bus.nodes[0]

def _next_load_id(node):
    """Find next available 1-char load ID for this node."""
    existing = set(node._load_map.keys())
    for c in [str(i) for i in range(1, 10)] + list('abcdefghijklmnopqrstuvwxyz'):
        if c not in existing:
            return c
    return None

def _remove_all_loads(s):
    """Remove all loads from a substation."""
    for old_load in list(s.loads):
        old_node = old_load.node
        if old_load.id in old_node._load_map:
            del old_node._load_map[old_load.id]

def _create_load(node, mw, q):
    """Create a new Load on node with given MW and Q."""
    lid = _next_load_id(node)
    if lid is None:
        return None
    new_load = Load(node, lid)
    new_load.status = True
    new_load.p = mw
    new_load.ps = mw
    new_load.q = q
    new_load.qs = q
    return new_load

MIN_345_LOAD_MW = 50

SPLIT_FRACTION = {
    69:  0.00,
    138: 0.012,
    345: 0.167,
}

MW_SPLIT = {
    138: {138: 0.97, 69: 0.03},
    345: {345: 0.64, 138: 0.36},
}

n_single = 0
n_split = 0
n_downgraded = 0

# Collect subs with load, grouped by max kV bucket
subs_with_load = [(s, sum(l.p for l in s.loads), sum(l.q for l in s.loads))
                  for s in subs if sum(l.p for l in s.loads) > 0]

for sub_bucket in [69, 138, 345]:
    bucket_subs = [(s, mw, q) for s, mw, q in subs_with_load
                   if _bucket_kv(max(b.nominal_kv for b in s.buses)) == sub_bucket]
    bucket_subs.sort(key=lambda x: -x[1])

    n_to_split = int(round(len(bucket_subs) * SPLIT_FRACTION.get(sub_bucket, 0)))
    split_ratios = MW_SPLIT.get(sub_bucket, {})

    for idx, (s, total_mw, total_q) in enumerate(bucket_subs):
        should_split = idx < n_to_split and len(split_ratios) > 0

        if should_split:
            _remove_all_loads(s)
            for target_kv, frac in split_ratios.items():
                mw_share = total_mw * frac
                q_share = total_q * frac
                if mw_share < 0.01:
                    continue
                if target_kv == 345 and mw_share < MIN_345_LOAD_MW:
                    target_kv = 138
                    n_downgraded += 1
                target_bus = _find_bus(s, target_kv)
                _create_load(_ensure_node(target_bus), mw_share, q_share)
            n_split += 1

        else:
            # Single kV: determine target
            if sub_bucket == 345 and total_mw < MIN_345_LOAD_MW:
                target_kv = 138
                n_downgraded += 1
            else:
                target_kv = sub_bucket

            target_bus = _find_bus(s, target_kv)

            # Skip if load is already on the correct bus
            if all(l.node.bus == target_bus for l in s.loads):
                n_single += 1
                continue

            _remove_all_loads(s)
            _create_load(_ensure_node(target_bus), total_mw, total_q)
            n_single += 1

print(f"Load voltage allocation: {n_single} single-kV, {n_split} split (highest-MW)")
print(f"  {n_downgraded} loads moved from 345 kV to 138 kV (below {MIN_345_LOAD_MW} MW threshold)")

# ── Split loads on parallel buses ───────────────────────────────────────
# Subs marked with _split_load have a duplicate kV level (parallel bus).
# Split load in half: one half stays on original bus, other goes to new bus.
n_parallel_split = 0
for s in subs:
    if not getattr(s, '_split_load', False):
        continue
    if len(s.buses) < 2:
        continue

    total_mw = sum(l.p for l in s.loads)
    total_q = sum(l.q for l in s.loads)
    if total_mw <= 0:
        continue

    buses_with_load = [b for b in s.buses if len(b.loads) > 0]
    buses_without_load = [b for b in s.buses if len(b.loads) == 0]

    if not buses_with_load or not buses_without_load:
        continue

    orig_bus = buses_with_load[0]
    new_bus = buses_without_load[-1]

    _remove_all_loads(s)
    half_mw = total_mw / 2.0
    half_q = total_q / 2.0
    _create_load(_ensure_node(orig_bus), half_mw, half_q)
    _create_load(_ensure_node(new_bus), half_mw, half_q)
    n_parallel_split += 1

print(f"Split {n_parallel_split} loads across parallel buses")


# ── 2031 ERCOT CDR Large Load Target (MW) ────────────────────────────────────
# ── Large Load Summary ─────────────────────────────────────────────────
print(f"Large loads: {len(synthetic_large_loads)} loads, {synthetic_large_loads['AssignedMW'].sum():,.0f} MW total")

# ── Create Dedicated Large-Load Substations ───────────────────────────────────
# Each synthetic large load gets its own substation (not attached to existing ones).
# These are NOT counted toward the core sub target (~4,700).
from gridworkbench.containers import Sub

n_large_loads_created = 0
ll_county_count = {}  # track naming per county

# Pre-build zone_number map; assert zone_dict exists
assert hasattr(sgb, 'zone_dict') and sgb.zone_dict, \
"sgb.zone_dict must exist before creating large-load subs (set in county assignment cell)"
zone_number_map_ll = {name: idx for idx, name in enumerate(sorted(sgb.zone_dict.keys()), 1)}
print(f"Zone map: {len(zone_number_map_ll)} counties")

sub_num_counter = max(s.number for s in sgb.wb.subs) + 1

for _, ll_row in synthetic_large_loads.iterrows():
    lat = float(ll_row['Latitude'])
    lon = float(ll_row['Longitude'])
    mw  = float(ll_row['AssignedMW'])

    # Assign area and zone
    wz = assign_weather_zone(lat, lon)
    if wz is None:
        continue
    county = assign_county(lat, lon)

    # Find the gridworkbench Area object for this weather zone
    area_obj = None
    for a in sgb.wb.areas:
        if a.name == wz:
            area_obj = a
            break
    if area_obj is None:
        continue

    # Determine kV based on MW tier and available levels
    if county and county in kv_zones:
        available_kvs = kv_zones[county]
    elif wz in kv_areas:
        available_kvs = kv_areas[wz]
    else:
        available_kvs = [138]

    # MW-based voltage tier (matches benchmark case patterns):
    # 75-300 MW → base_kv (typically 138 kV)
    # 300+ MW → 345 kV if available in the area, else highest available <= 345
    base_kv = available_kvs[0]
    if mw >= 300:
        # Pick 345 kV if available, else the highest kV <= 345
        candidates_345 = [k for k in available_kvs if k <= 345]
        ll_kv = max(candidates_345) if candidates_345 else base_kv
    else:
        ll_kv = base_kv

    # Create unique name: LL_CountyName_N
    county_label = county if county else wz.replace(' ', '')
    ll_county_count[county_label] = ll_county_count.get(county_label, 0) + 1
    sub_name = f"LL_{county_label}_{ll_county_count[county_label]}"

    # Create new Sub
    ll_sub = Sub(area_obj, sub_num_counter)
    sub_num_counter += 1
    ll_sub.name = sub_name
    ll_sub.latitude = lat
    ll_sub.longitude = lon

    # Create Bus + Node + Load
    ll_bus = Bus(ll_sub, busnum)
    busnum += 1
    ll_bus.nominal_kv = ll_kv
    ll_bus.name = f"{sub_name}_{ll_kv}"
    # Assign zone number for county
    if county and county in zone_number_map_ll:
        ll_bus.zone_number = zone_number_map_ll[county]
    else:
        print(f"  WARNING: No zone for county '{county}' at {sub_name}, defaulting to 1")
        ll_bus.zone_number = 1

    ll_node = Node(ll_bus, ll_bus.number)
    ll_node.name = ll_bus.name

    ll_load = Load(ll_node, "1")
    ll_load.status = True
    ll_load.ps = ll_load.p = mw
    ll_load.qs = ll_load.q = mw * 0.33  # PF ~ 0.95

    n_large_loads_created += 1

print(f"\nCreated {n_large_loads_created} dedicated large-load substations (>= 75 MW)")
large_load_total_mw = sum(sum(l.p for l in s.loads) for s in sgb.wb.subs if s.name.startswith('LL_'))
print(f"Total large load MW: {large_load_total_mw:,.0f} MW")

# Print large load voltage distribution
ll_kv_dist = {}
for s_ll in sgb.wb.subs:
    if not s_ll.name.startswith('LL_'):
        continue
    for b in s_ll.buses:
        ll_kv_dist[b.nominal_kv] = ll_kv_dist.get(b.nominal_kv, 0) + 1
print(f"Large load kV distribution: {dict(sorted(ll_kv_dist.items()))}")

# ── Label Large Loads: DataCenter (IM3 proximity) vs LargeLoad (generic) ──────
# Only subs within DC_PROXIMITY_KM of a real TX data center (IM3 atlas) get
# the _DataCenter suffix. All others are labeled _LargeLoad.
# This ensures DataCenter labels reflect actual data center geography.
DC_PROXIMITY_KM = 50

dc_atlas = pd.read_csv(os.path.join(sgb.input_folder, 'im3_open_source_data_center_atlas.csv'))
dc_texas = dc_atlas[dc_atlas['state_abb'] == 'TX'].copy()
print(f"\nTexas data centers from IM3 atlas: {len(dc_texas)}")

def _haversine_deg(lat1, lon1, lat2, lon2):
    from math import radians, cos, sin, asin, sqrt
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return 2 * 6371 * asin(sqrt(a))

dc_lats = dc_texas['lat'].values
dc_lons = dc_texas['lon'].values

ll_subs_list = [s_ll for s_ll in sgb.wb.subs if s_ll.name.startswith('LL_')]
n_dc = 0
n_ll = 0
dc_mw = 0.0
ll_mw = 0.0

for s_ll in ll_subs_list:
    dists = [_haversine_deg(s_ll.latitude, s_ll.longitude, dc_lats[i], dc_lons[i])
             for i in range(len(dc_lats))]
    min_dist = min(dists) if dists else 9999
    mw = sum(l.p for l in s_ll.loads)

    if min_dist <= DC_PROXIMITY_KM:
        suffix = '_DataCenter'
        n_dc += 1
        dc_mw += mw
    else:
        suffix = '_LargeLoad'
        n_ll += 1
        ll_mw += mw

    s_ll.name = s_ll.name + suffix
    for b in s_ll.buses:
        b.name = b.name + suffix

total_mw = dc_mw + ll_mw
print(f"  DataCenter:  {n_dc:>4} subs  {dc_mw:>10,.0f} MW ({100*dc_mw/total_mw:.1f}%)")
print(f"  LargeLoad:   {n_ll:>4} subs  {ll_mw:>10,.0f} MW ({100*ll_mw/total_mw:.1f}%)")
print(f"  Total:       {n_dc+n_ll:>4} subs  {total_mw:>10,.0f} MW")

# ── Scale Census Loads to ~145 GW Target (per-zone weighted) ─────────────────
# Census loads are scaled per weather zone to match the benchmark case distribution.
# Large loads keep their benchmark MW sizes (not scaled).
# Small per-zone noise (±3%) makes the total realistic rather than exact.
TARGET_SYSTEM_LOAD_MW = 145000  # ERCOT 2031 CDR forecast

census_load_target_mw = TARGET_SYSTEM_LOAD_MW - large_load_total_mw

# Compute per-zone raw census load totals (before scaling)
zone_raw_mw = {}
for s in subs:
    wz = s.area.name
    zone_raw_mw[wz] = zone_raw_mw.get(wz, 0) + sum(l.p for l in s.loads)
total_raw_mw = sum(zone_raw_mw.values())

# Per-zone scale factor: zone_target / zone_raw_mw
# zone_target = census_load_target_mw * ref_zone_load_share (from benchmark case)
# Add ±3% per-zone noise so the total is realistic, not a round number
rng = np.random.default_rng(42)
zone_scale = {}
for wz in zone_raw_mw:
    ref_share = ref_zone_load_shares.get(wz, 1.0 / len(zone_raw_mw))
    zone_target = census_load_target_mw * ref_share
    noise = 1.0 + rng.uniform(-0.03, 0.03)
    zone_scale[wz] = (zone_target / zone_raw_mw[wz] * noise) if zone_raw_mw[wz] > 0 else 1.0

# Apply per-zone scaling to census-based loads (core subs only)
for s in subs:
    wz = s.area.name
    scale = zone_scale.get(wz, 1.0)
    for l in s.loads:
        l.ps = l.p = l.p * scale
        l.qs = l.q = l.q * scale

# Print load scaling summary
print(f"\nLoad scaling to ~{TARGET_SYSTEM_LOAD_MW:,} MW (census + large, with ±3% zone noise):")
print(f"  Large load total:    {large_load_total_mw:>10,.0f} MW ({100*large_load_total_mw/TARGET_SYSTEM_LOAD_MW:.1f}%)")
print(f"  Census load target:  {census_load_target_mw:>10,.0f} MW ({100*census_load_target_mw/TARGET_SYSTEM_LOAD_MW:.1f}%)")
print(f"  {'Zone':<20s} {'Raw MW':>10s} {'Scaled MW':>10s} {'Factor':>8s} {'Ref Share':>10s}")
scaled_total = 0
for wz in sorted(zone_scale):
    scaled_mw = zone_raw_mw[wz] * zone_scale[wz]
    scaled_total += scaled_mw
    print(f"  {wz:<20s} {zone_raw_mw[wz]:>10.0f} {scaled_mw:>10.0f} {zone_scale[wz]:>8.2f}x {ref_zone_load_shares.get(wz,0):>9.1%}")
print(f"  {'TOTAL':<20s} {total_raw_mw:>10.0f} {scaled_total:>10.0f}")
actual_system_mw = sum(sum(l.p for l in s.loads) for s in sgb.wb.subs)
print(f"  System total: {actual_system_mw:,.0f} MW (target: ~{TARGET_SYSTEM_LOAD_MW:,})")

# ── Summary Statistics ────────────────────────────────────────────────────────
all_subs = sgb.wb.subs
all_buses = sgb.wb.buses
n_core = len(subs)
n_ll = n_large_loads_created
n_ehv_final = sum(1 for s in all_subs if max(b.nominal_kv for b in s.buses) > 200)

# Count loads per kV level
kv_load_count = {}
kv_load_mw = {}
for s in all_subs:
    for b in s.buses:
        kv = b.nominal_kv
        n_loads = len(b.loads)
        mw_loads = sum(l.p for l in b.loads)
        kv_load_count[kv] = kv_load_count.get(kv, 0) + n_loads
        kv_load_mw[kv] = kv_load_mw.get(kv, 0) + mw_loads

# Count DataCenter subs
n_dc = sum(1 for s in all_subs if '_DataCenter' in s.name)
dc_total_mw = sum(sum(l.p for l in s.loads) for s in all_subs if '_DataCenter' in s.name)

print(f"\n{'='*60}")
print(f"SUBSTATION PLANNING SUMMARY")
print(f"{'='*60}")
print(f"Core substations (load/gen):     {n_core}")
print(f"Large-load substations:          {n_ll}")
print(f"  of which DataCenter:           {n_dc} ({dc_total_mw:,.0f} MW)")
print(f"Total substations:               {len(all_subs)}")
print(f"EHV substations (>200kV):        {n_ehv_final} ({100*n_ehv_final/n_core:.1f}% of core)")
print(f"Total buses:                     {len(all_buses)}")
print(f"Avg buses per sub:               {len(all_buses)/len(all_subs):.2f}")
actual_total_mw = sum(sum(l.p for l in s.loads) for s in all_subs)
print(f"Total system load:               {actual_total_mw:,.0f} MW")
print(f"\nLoads per kV level:")
for kv in sorted(kv_load_count):
    print(f"  {kv:>6} kV: {kv_load_count[kv]:>5} loads, {kv_load_mw[kv]:>10,.0f} MW")
print(f"{'='*60}")
print(f"Note: switching subs will be added during transmission planning")

# ── Push cluster # and EIA 860 data to custom fields ──────────────────────────
sgb.wb.pw_instructions["sub.cluster"] = {"pwfield": ["CustomInteger:0"],
            "import_from_aux": "int"}
sgb.wb.pw_instructions["gen.eia860_plant"] = {"pwfield": ["CustomString:0"],
            "import_from_aux": "string"}
sgb.wb.pw_instructions["gen.eia860_generator"] = {"pwfield": ["CustomString:1"],
            "import_from_aux": "string"}

AUX_OUT = os.path.join(OUTPUT_DIR, "ERCOT_Subs.aux")
sgb.export_aux(AUX_OUT)

# ── Append Zone name records to AUX file ──────────────────────────────────────
if hasattr(sgb, 'zone_dict') and sgb.zone_dict:
    zone_number_map = {name: idx for idx, name in enumerate(sorted(sgb.zone_dict.keys()), 1)}
    with open(AUX_OUT, "a") as f:
        f.write("\nDATA (Zone, [ZoneNum, ZoneName])\n{\n")
        for zone_name in sorted(sgb.zone_dict.keys()):
            znum = zone_number_map[zone_name]
            f.write(f'{znum} "{zone_name}"\n')
        f.write("}\n")
    print(f"Appended {len(zone_number_map)} zone names (counties) to AUX file")

lap(f"Exported ERCOT_Subs.aux → {AUX_OUT}")



In [ ]:
# ── Presentation Figures (saved individually) ────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
import numpy as np

load_mw = [sum(l.p for l in s.loads) for s in subs]
lats = [s.latitude for s in subs]
lons = [s.longitude for s in subs]
positive_loads = [m for m in load_mw if m > 0]
vmin = max(1, min(positive_loads)) if positive_loads else 1
vmax = max(load_mw) if max(load_mw) > 0 else 1

# ── Figure 1: Load Distribution with 765 kV Subs ────────────────────────────
fig1, ax = plt.subplots(figsize=(10, 8))
if 'ercot_zones_gdf' in dir():
    ercot_zones_gdf.boundary.plot(ax=ax, color='gray', linewidth=0.8)
sc = ax.scatter(lons, lats, s=3, c=[max(m, 0.1) for m in load_mw],
                cmap='YlOrRd', alpha=0.7,
                norm=mcolors.LogNorm(vmin=vmin, vmax=vmax))
cbar = plt.colorbar(sc, ax=ax, shrink=0.7, pad=0.02)
cbar.set_label('Load (MW)', fontsize=10)
for s, route, idx, d in route_assignments:
    color = 'darkorange' if 'Permian' in route else 'deeppink'
    ax.scatter(s.longitude, s.latitude, s=500, c=color, marker='*',
               edgecolors='black', linewidths=1.0, zorder=5)
ax.set_title('Load Distribution with Proposed 765 kV Substations', fontsize=12, fontweight='bold')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude'); ax.set_aspect('equal')
plt.tight_layout()
fig1.savefig('765kV_design_overview_panel1.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: panel1')

# ── Figure 2: 765 kV Route Map ──────────────────────────────────────────────
fig2, ax = plt.subplots(figsize=(10, 8))
if 'ercot_zones_gdf' in dir():
    ercot_zones_gdf.plot(ax=ax, color='#f0f0f0', edgecolor='gray', linewidth=0.8)
ax.scatter(lons, lats, s=1, c='lightgray', alpha=0.2)
permian_subs = [(s, route, idx) for s, route, idx, d in route_assignments if 'Permian' in route]
eastern_subs = [(s, route, idx) for s, route, idx, d in route_assignments if 'Eastern' in route]
for s, route, idx in permian_subs:
    ax.scatter(s.longitude, s.latitude, s=300, c='darkorange', marker='*',
               edgecolors='black', linewidths=0.8, zorder=5)
    ax.annotate(f'{idx+1}', (s.longitude, s.latitude), fontsize=8, fontweight='bold',
                xytext=(6, 6), textcoords='offset points', color='darkorange')
for s, route, idx in eastern_subs:
    ax.scatter(s.longitude, s.latitude, s=300, c='deeppink', marker='*',
               edgecolors='black', linewidths=0.8, zorder=5)
    ax.annotate(f'{idx+1}', (s.longitude, s.latitude), fontsize=8, fontweight='bold',
                xytext=(6, 6), textcoords='offset points', color='deeppink')
legend_elements = [
    Line2D([0],[0], marker='*', color='w', markerfacecolor='darkorange', markersize=14,
           markeredgecolor='black', label=f'Permian Basin Route ({len(permian_subs)} subs)'),
    Line2D([0],[0], marker='*', color='w', markerfacecolor='deeppink', markersize=14,
           markeredgecolor='black', label=f'Eastern Route ({len(eastern_subs)} subs)'),
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=10, framealpha=0.9)
ax.set_title('Proposed 765 kV Substation Locations', fontsize=12, fontweight='bold')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude'); ax.set_aspect('equal')
plt.tight_layout()
fig2.savefig('765kV_design_overview_panel2.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: panel2')

# ── Figure 3: Large Load Centers by Type ──────────────────────────────────────
fig3, ax = plt.subplots(figsize=(10, 8))
if 'ercot_zones_gdf' in dir():
    ercot_zones_gdf.plot(ax=ax, color='#f0f0f0', edgecolor='gray', linewidth=0.8)
ax.scatter(lons, lats, s=1, c='lightgray', alpha=0.2)

# Large load type colors (IM3-based: DataCenter vs generic LargeLoad)
ll_type_colors = {
    'DataCenter': 'crimson',
    'LargeLoad': 'dodgerblue',
}

ll_subs = [s for s in sgb.wb.subs if hasattr(s, 'name') and s.name.startswith('LL_')]
plotted_types = set()
for s in ll_subs:
    if '_DataCenter' in s.name:
        ltype, color = 'DataCenter', 'crimson'
    else:
        ltype, color = 'LargeLoad', 'dodgerblue'
    mw = sum(l.p for l in s.loads)
    ax.scatter(s.longitude, s.latitude, s=max(20, mw/20), c=color, alpha=0.7,
               edgecolors='black', linewidths=0.3, zorder=4)
    plotted_types.add((ltype, color))

# 765 kV substations
for s, route, idx, d in route_assignments:
    ax.scatter(s.longitude, s.latitude, s=400, c='gold', marker='*',
               edgecolors='black', linewidths=1.0, zorder=5)

ll_legend = [Line2D([0],[0], marker='o', color='w', markerfacecolor=c, markersize=8,
                     markeredgecolor='black', label=t) for t, c in sorted(plotted_types)]
ll_legend.append(Line2D([0],[0], marker='*', color='w', markerfacecolor='gold',
                         markersize=14, markeredgecolor='black', label='765 kV Sub'))
ax.legend(handles=ll_legend, loc='upper right', fontsize=9, framealpha=0.9)
ax.set_title('Large Load Centers by Type and 765 kV Substations', fontsize=12, fontweight='bold')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude'); ax.set_aspect('equal')
plt.tight_layout()
fig3.savefig('765kV_design_overview_panel3.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: panel3')

# ── Figure 4: Voltage Level Distribution ─────────────────────────────────────
fig4, ax = plt.subplots(figsize=(10, 8))
if 'ercot_zones_gdf' in dir():
    ercot_zones_gdf.plot(ax=ax, color='#f0f0f0', edgecolor='gray', linewidth=0.8)
kv_colors = {69: 'limegreen', 138: 'royalblue', 345: 'darkorange', 765: 'magenta'}
kv_sizes = {69: 6, 138: 6, 345: 15, 765: 80}
for kv in sorted(kv_colors.keys()):
    kv_subs_list = [s for s in subs if kv in s.kv_levels]
    if kv_subs_list:
        ax.scatter([s.longitude for s in kv_subs_list],
                   [s.latitude for s in kv_subs_list],
                   s=kv_sizes[kv], c=kv_colors[kv], alpha=0.7,
                   label=f'{kv} kV ({len(kv_subs_list)} buses)',
                   zorder=2 + list(kv_colors.keys()).index(kv))
ax.legend(loc='upper right', fontsize=10, framealpha=0.9)
ax.set_title('Voltage Level Distribution', fontsize=12, fontweight='bold')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude'); ax.set_aspect('equal')
plt.tight_layout()
fig4.savefig('765kV_design_overview_panel4.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: panel4')

# Cell 16: Bus Creation, Load Voltage Allocation, Large-Load Subs, and Export

Final cell — builds buses, allocates loads to voltage levels based on reference case statistics, creates large-load substations, scales census load, and exports.

## Load Voltage Assignment Rules

Loads are assigned to bus voltage levels based on statistics from the reference case:

**Load Voltage Distribution (Reference Case)**

| Sub Max kV | → 69 kV | → 138 kV | → 345 kV |
|------------|---------|----------|----------|
| 69 | 100% | 0% | 0% |
| 138 | 3% | 97% | 0% |
| 345 | 3% | 68% | 30% |

**Load Size by Voltage Level (Reference Case)**

| Bus kV | Median (MW) | Mean (MW) | Max (MW) | Description |
|--------|-------------|-----------|----------|-------------|
| 69 | 3.2 | 5.6 | 232 | Small distribution loads |
| 138 | 14.8 | 19.3 | 850 | Bulk system loads |
| 345 | 20.0 | 118.2 | 1,400 | Large industrial / data center |

> Loads below 50 MW at 345 kV substations are placed on the 138 kV bus instead. Only the highest-MW substations (top 1.2% for 138 kV, top 16.7% for 345 kV) get loads split across two voltage levels. Parallel buses split their load 50/50 across two buses at the same kV.

## Tunable Parameters

| Variable | Default | Effect |
|----------|---------|--------|
| `MIN_345_LOAD_MW` | 50 | Minimum MW for a load on a 345 kV bus. Below this → 138 kV. |
| `SPLIT_FRACTION` | 138: 1.2%, 345: 16.7% | Fraction of highest-MW subs with split loads across kV levels. |
| `DC_PROXIMITY_KM` | 50 | Max distance (km) to label a large-load sub as DataCenter. |
| `TARGET_SYSTEM_LOAD_MW` | 145,000 | Total system load target (2031 ERCOT CDR). |

## Pipeline Steps

1. **Bus creation** — creates Bus objects at each substation, one per kV level (including parallel buses)
2. **Generator voltage assignment** — moves large generators (>= 75 MW) to highest-voltage bus
3. **Load voltage allocation** — assigns loads based on reference case statistics
4. **Parallel bus load split** — splits load 50/50 on subs with parallel buses
5. **Large-load substations** — creates dedicated `LL_{County}_{N}` subs
6. **DataCenter labeling** — labels large-load subs within 50 km of IM3 atlas data centers
7. **Census load scaling** — scales per-zone census loads to reach target system load
8. **AUX export** — saves `ERCOT_Subs.aux` with zone name records
